In [3]:
from pathlib import Path
from config import DATA_PATH_PROCESSED, IMAGES_SUBDIR, LABELS_SUBDIR, THERMAL_SUBDIR, ANNOTATED_IMAGES_SUBDIR

SPLITS = ["train", "val", "test"]

# class mapping
CLASS_NAMES = {
    0: "animal",
    # 1: "red_deer",
    # 2: "roe_deer",
    # 3: "chamois",
    # 4: "human",
    # 5: "alpine_ibex",
    # 6: "fallow_deer",
    # 7: "unknown",
    # 8: "dog",
    # 9: "bird",
    # 10: "wild_boar",
    # 11: "hybrid_pig"
}

In [2]:
from pathlib import Path
from collections import Counter
import numpy as np
import csv

images_total = 0
images_with_animals = 0
images_without_animals = 0

animals_per_image = []

class_counter = Counter()

In [4]:
info ={split: 0 for split in SPLITS}
for split in SPLITS:
    label_dir = DATA_PATH_PROCESSED / ANNOTATED_IMAGES_SUBDIR / LABELS_SUBDIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        images_total += 1

        with open(file, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # Case: empty or only "0" → no animal image
        if len(lines) == 0:
            images_without_animals += 1
            info[split] += 1
            animals_per_image.append(0)
            continue

        # image contains animals
        images_with_animals += 1
        animals_per_image.append(len(lines))

        for line in lines:
            cls = int(line.split()[0])


            class_counter[cls] += 1

total_animals = sum(class_counter.values())
avg_animals_per_image = total_animals / images_total if images_total else 0

animals_array = np.array(animals_per_image)

In [5]:
print(f"Total images: {images_total}")
print(f"Images with animals: {images_with_animals}")
print(f"Images without animals: {images_without_animals}")
print(f"Images without animals per split: {info}")
print(f"Percentage with animals: {images_with_animals / images_total * 100:.2f}%")

print("\n--- Animal stats ---")
print(f"Total animals: {total_animals}")
print(f"Average animals per image: {avg_animals_per_image:.2f}")
print(f"Max animals in one image: {animals_array.max()}")

print(f"95th percentile animals/image: {np.percentile(animals_array, 95):.2f}")
print(f"99th percentile animals/image: {np.percentile(animals_array, 99):.2f}")


print("\n--- Per class distribution ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    name = CLASS_NAMES[cls_id]
    count = class_counter.get(cls_id, 0)

    print(f"{cls_id:2d} {name:15s}: {count}")


Total images: 9179
Images with animals: 8264
Images without animals: 915
Images without animals per split: {'train': 855, 'val': 60, 'test': 0}
Percentage with animals: 90.03%

--- Animal stats ---
Total animals: 26865
Average animals per image: 2.93
Max animals in one image: 26
95th percentile animals/image: 10.00
99th percentile animals/image: 17.00

--- Per class distribution ---
 0 animal         : 26865


### Class occurrences per split

Compare box counts per class across train / val / test. A large mismatch between splits (e.g. a class only in val) will hurt training and make mAP misleading.

In [6]:
class_counter_per_split = {split: Counter() for split in SPLITS}

for split in SPLITS:
    label_dir = DATA_PATH_PROCESSED / ANNOTATED_IMAGES_SUBDIR / LABELS_SUBDIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls = int(line.split()[0])
                class_counter_per_split[split][cls] += 1

# table: box counts per class per split
print("--- Per-class occurrences per split (box counts) ---\n")
col_w = 10
header = f"{'id':>3} {'class':15s}" + "".join(f"{s:>{col_w}s}" for s in SPLITS)
print(header)
print("-" * len(header))

for cls_id in sorted(CLASS_NAMES.keys()):
    counts = [class_counter_per_split[s].get(cls_id, 0) for s in SPLITS]
    if sum(counts) == 0:
        continue
    print(
        f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
        + "".join(f"{c:>{col_w}d}" for c in counts)
    )

print("-" * len(header))
split_totals = [sum(class_counter_per_split[s].values()) for s in SPLITS]
print(f"{'':3} {'TOTAL':15s}" + "".join(f"{t:>{col_w}d}" for t in split_totals))

# # percentage within each split -> with 1 class not usefule anymore
# print("\n--- Per-class share within each split (%) ---\n")
# print(header)
# print("-" * len(header))

# for cls_id in sorted(CLASS_NAMES.keys()):
#     pcts = []
#     has_any = False
#     for split in SPLITS:
#         total = split_totals[SPLITS.index(split)] or 1
#         count = class_counter_per_split[split].get(cls_id, 0)
#         pcts.append(count / total * 100)
#         has_any |= count > 0
#     if not has_any:
#         continue
#     print(
#         f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
#         + "".join(f"{p:>{col_w}.1f}" for p in pcts)
#     )

--- Per-class occurrences per split (box counts) ---

 id class               train       val      test
-------------------------------------------------
  0 animal              24177      2688         0
-------------------------------------------------
    TOTAL               24177      2688         0


In [7]:
#1 class -> not usefule anmyore
print("\n--- Class imbalance (percentage, FULL) ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    print(f"{CLASS_NAMES[cls_id]:15s}: {pct:.2f}%")


--- Class imbalance (percentage, FULL) ---
animal         : 100.00%


In [8]:
#1 class -> not usefule anmyore

print("\n--- Rare classes (<5%) ---")

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    if pct < 5:
        print(f"{CLASS_NAMES[cls_id]:15s}: {count:5d} - {pct:.2f}%")


--- Rare classes (<5%) ---


In [9]:
with open("analysis/dataset_class_distribution_preprocessed.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "class_name", "count", "percentage"])

    for cls_id, count in sorted(class_counter.items()):
        pct = (count / total_animals) * 100 if total_animals else 0
        writer.writerow([cls_id, CLASS_NAMES.get(cls_id), count, pct])

print("\nCSV saved: dataset_class_distribution_preprocessed.csv")


CSV saved: dataset_class_distribution_preprocessed.csv
